# 01 — Dataset design review

**Objectives**
1. Inspect the seven-table data model from the schema registry (the single source of truth).
2. Review grain, keys, relationships and allowed values.
3. See join fan-out and the net-sales definition on a tiny worked example with hand-checkable numbers.
4. Run assertions that the design is internally consistent.

**Prerequisites:** notebook 00 passes. **Configuration:** none. **Reads:** only `src/` (no generated data needed).

**Design documents:** `docs/02_dataset_specification.md` (ERD, metrics, reconciliation) and `docs/data_dictionary.md`.

In [1]:
# Locate the project root (folder containing pyproject.toml) so the notebook works
# no matter which folder VS Code starts the kernel in, then make `src/` importable.
import os, sys
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    if root.parent == root:
        raise RuntimeError("Could not find pyproject.toml - open the analytics-agent folder in VS Code.")
    root = root.parent
os.chdir(root)
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
print("Project root:", root)
print("Python:", sys.version.split()[0], "| executable:", sys.executable)

Project root: /home/claude/analytics-agent
Python: 3.12.3 | executable: /home/claude/.venv/bin/python


In [2]:
import pandas as pd
from analytics_agent import schema_registry as sr
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## Step 1 — Table overview: purpose, grain and keys

Why: every metric error starts with confusing grain. **Checkpoint:** 7 rows, each with a primary key.

In [3]:
overview = pd.DataFrame([{
    "table": t.name, "grain": t.grain, "primary_key": ", ".join(t.primary_key),
    "other_unique": "; ".join("(" + ", ".join(k) + ")" for k in t.business_keys),
    "columns": len(t.columns), "expected_rows": t.expected_rows,
} for t in sr.TABLES.values()])
overview

,table,grain,primary_key,other_unique,columns,expected_rows
0,customers,One row per customer_id.,customer_id,(email),7,"~2,000 (clean)"
1,products,One row per product_id.,product_id,(sku),8,~250 (clean)
2,orders,One row per order_id.,order_id,,13,"~15,000 (clean)"
3,order_items,One row per order_item_id (one product line within an order).,order_item_id,"(order_id, line_number); (order_id, product_id)",9,"~33,000 (clean)"
4,payments,One row per payment attempt (payment_id). An order can have several attempts.,payment_id,(provider_transaction_id),8,"~15,500 (clean)"
5,shipments,One row per shipment_id. In this version an order has at most one shipment; ...,shipment_id,(order_id),7,"~13,800 (clean)"
6,refunds,One row per refund transaction (refund_id).,refund_id,,8,"~1,700 (clean)"


## Step 2 — Relationships (foreign keys)

**Checkpoint:** every FK points to a table loaded earlier (parents before children).

In [4]:
fks = pd.DataFrame([{"child": t.name, "column": fk.column, "parent": f"{fk.ref_table}.{fk.ref_column}",
                     "nullable": fk.nullable} for t in sr.TABLES.values() for fk in t.foreign_keys])
pos = {n: i for i, n in enumerate(sr.TABLE_LOAD_ORDER)}
assert all(pos[p.split(".")[0]] < pos[c] for c, p in zip(fks.child, fks.parent))
fks

,child,column,parent,nullable
0,orders,customer_id,customers.customer_id,False
1,order_items,order_id,orders.order_id,False
2,order_items,product_id,products.product_id,False
3,payments,order_id,orders.order_id,False
4,shipments,order_id,orders.order_id,False
5,refunds,order_id,orders.order_id,False
6,refunds,order_item_id,order_items.order_item_id,True
7,refunds,payment_id,payments.payment_id,False


## Step 3 — Column detail for one table

Change `TABLE` to inspect any table. The complete dictionary is `docs/data_dictionary.md`.

In [5]:
TABLE = "orders"
t = sr.get_table(TABLE)
pd.DataFrame([{"column": c.name, "type": c.bq_type, "mode": c.mode,
               "allowed": ", ".join(c.allowed_values) if c.allowed_values else "",
               "pii": c.pii, "description": c.description} for c in t.columns])

,column,type,mode,allowed,pii,description
0,order_id,STRING,REQUIRED,,False,"Surrogate order identifier, format ORD0000001."
1,customer_id,STRING,REQUIRED,,False,FK to customers.customer_id.
2,order_timestamp,TIMESTAMP,REQUIRED,,False,When the order was placed (UTC).
3,order_date,DATE,REQUIRED,,False,DATE(order_timestamp) in UTC. Partition column and the date used to attribut...
4,order_status,STRING,REQUIRED,"pending_payment, processing, shipped, delivered, cancelled",False,Lifecycle status as of the data snapshot. Refund state is NOT stored here; s...
5,sales_channel,STRING,REQUIRED,"web, mobile_app, marketplace",False,Channel the order was placed through.
6,currency_code,STRING,REQUIRED,USD,False,Currency of all money columns.
7,items_subtotal,NUMERIC,REQUIRED,,False,SUM(order_items.gross_amount) for the order (before discounts).
8,discount_amount,NUMERIC,REQUIRED,,False,SUM(order_items.discount_amount) for the order.
9,shipping_amount,NUMERIC,REQUIRED,,False,"Shipping charged to the customer (0.00 when merchandise net >= 75.00, else 6..."


## Step 4 — Join fan-out warnings recorded in the registry

In [6]:
for t in sr.TABLES.values():
    if t.fanout_warning:
        print(f"{t.name}: {t.fanout_warning}\n")

orders: Joining orders to order_items, payments, or refunds multiplies order rows. Never SUM order-level columns (order_total, shipping_amount, tax_amount) after such a join; aggregate the child table first, or use COUNT(DISTINCT order_id).

payments: Orders can have a failed attempt before the captured one. Filter payment_status = 'captured' and aggregate before joining to orders.



## Step 5 — Worked example: why fan-out breaks totals

Two booked orders: order A has two lines, order B one line. Summing the order-level `order_total` **after** joining to lines counts order A twice. This is wrong answer **W01** in the gold set.

In [7]:
orders = pd.DataFrame({"order_id": ["A", "B"], "order_total": [48.03, 28.19]})
items = pd.DataFrame({"order_item_id": ["A1", "A2", "B1"], "order_id": ["A", "A", "B"], "net_amount": [20.0, 18.0, 20.0]})
joined = items.merge(orders, on="order_id")
wrong = joined.order_total.sum()
right = orders.order_total.sum()
print(f"SUM(order_total) after join : {wrong:.2f}   <- order A counted twice")
print(f"SUM(order_total) per order  : {right:.2f}")
print(f"SUM(net_amount) from lines  : {joined.net_amount.sum():.2f}   <- item column, safe to sum")
assert round(wrong, 2) == 124.25 and round(right, 2) == 76.22

SUM(order_total) after join : 124.25   <- order A counted twice
SUM(order_total) per order  : 76.22
SUM(net_amount) from lines  : 58.00   <- item column, safe to sum


## Step 6 — Worked example: the net-sales definition

Net sales = merchandise after discounts on **booked** orders − **completed** item refunds. Tax, shipping, cancelled orders, pending-payment orders and pending refunds are excluded. The code below uses the real metric function on a five-order hand-built dataset (the same one used in `tests/unit/test_metrics.py`).

In [8]:
from analytics_agent.data_generation.tiny_dataset import build_tiny_dataset
from analytics_agent import metrics as m
tiny = build_tiny_dataset()
print(tiny["orders"][["order_id", "order_date", "order_status", "order_total"]].to_string(index=False))
print()
print(m.net_sales(tiny, "2026-01-01", "2026-02-28", group_by=["month"]).to_string(index=False))

order_id order_date    order_status  order_total
      O1 2026-01-10       delivered        48.03
      O2 2026-01-20       delivered        28.19
      O3 2026-02-05       cancelled        10.70
      O4 2026-02-10 pending_payment        17.59
      O5 2026-02-15         shipped        28.39

  month  merch_sales  refunds  net_sales
2026-01         58.0      5.0       53.0
2026-02         20.0      0.0       20.0


**Hand check:** January = 20 + 18 + 20 merchandise − 5.00 completed refund = **53.00**; February = O5 only = **20.00** (O3 cancelled, O4 pending payment, R2 pending refund all excluded).

In [9]:
ns = m.net_sales(tiny, "2026-01-01", "2026-02-28", group_by=["month"]).set_index("month").net_sales
assert ns["2026-01"] == 53.00 and ns["2026-02"] == 20.00
rec = m.reconciliation(tiny, "2026-01-01", "2026-02-28")
rec

{'merch_sales_booked': 78.0,
 'shipping_booked': 20.97,
 'tax_booked': 5.64,
 'captured_on_cancelled_orders': 10.7,
 'captured_payments': 115.31,
 'pending_payments': 17.59,
 'completed_item_refunds': 5.0,
 'completed_cancellation_refunds': 10.7,
 'pending_refunds_excluded': 20.0,
 'net_sales': 73.0,
 'net_cash': 99.61,
 'bridge_difference': 0.0}

**Reconciliation check:** captured 115.31 = merchandise 78.00 + shipping 20.97 + tax 5.64 + captured on cancelled 10.70, so `bridge_difference` is 0.00.

## Step 7 — Design assertions (same checks as `tests/unit/test_schema_registry.py`)

In [10]:
for t in sr.TABLES.values():
    assert len(set(t.column_names)) == len(t.column_names), t.name
    for col in t.primary_key:
        assert t.column(col).mode == "REQUIRED"
assert set(sr.BOOKED_ORDER_STATUSES) <= set(sr.ORDER_STATUSES)
assert sr.CUSTOMERS.column("email").pii
print("Design assertions passed.")

Design assertions passed.


## Troubleshooting
| Symptom | Fix |
|---|---|
| Import errors | Re-run the bootstrap cell; see notebook 00. |

## Summary and next steps
You reviewed grain, keys, relationships, fan-out and the net-sales definition. **Please review `docs/02_dataset_specification.md` before Stage 3** — table names and metric definitions are treated as approved interfaces from then on. Next: **02_generate_synthetic_data**.

## References
- BigQuery data types: https://cloud.google.com/bigquery/docs/reference/standard-sql/data-types
- BigQuery partitioned tables: https://cloud.google.com/bigquery/docs/partitioned-tables
- BigQuery clustered tables: https://cloud.google.com/bigquery/docs/clustered-tables